# 5.1 Length and Dot Product in $\mathbb{R}^n$

**Course:** Elementary Linear Algebra (Larson, 8e)  
**Chapter 5:** Inner Product Spaces

---

### What you will learn

- How to compute the **dot product** of two vectors and what it measures geometrically.
- How to compute the **norm** (length) of a vector and how to **normalize** it to a unit vector.
- How to measure the **distance** between two vectors and the **angle** between them.
- What **orthogonality** means and how to detect it algebraically.
- Why the **Cauchy–Schwarz inequality** and **triangle inequality** hold, and what they mean.
- How **cosine similarity** connects linear algebra to real-world search engines.

---

## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '../..'))

In [ ]:
from linalg_core.matrix import Matrix
from linalg_core.inner import dot, norm, normalize, distance, angle, are_orthogonal
from linalg_core.elimination import solve
from linalg_core import EPSILON

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import math
import random

random.seed(42)

print("Setup complete.")

---

## 2. Motivation — Search Engine Similarity

How does a search engine decide which document best matches your query?

One powerful technique treats both **documents** and **queries** as vectors.
Each dimension corresponds to a word (or term), and the entry counts how
often that term appears. The engine then ranks documents by the **cosine
of the angle** between the document vector and the query vector.

$$\text{cosine similarity} = \cos\theta = \frac{\mathbf{d} \cdot \mathbf{q}}{\|\mathbf{d}\|\,\|\mathbf{q}\|}$$

A cosine of 1 means the vectors point in the same direction (perfect match).
A cosine of 0 means they are orthogonal (no overlap at all).

**Example.** Suppose our vocabulary has four terms. A document and a query are:

$$\mathbf{d} = \begin{bmatrix} 3 \\ 1 \\ 0 \\ 2 \end{bmatrix}, \qquad
\mathbf{q} = \begin{bmatrix} 1 \\ 0 \\ 0 \\ 1 \end{bmatrix}$$

How similar are they? To answer this, we need the **dot product**, **norms**,
and **angle** — precisely the tools we build in this notebook.
We will return to this problem at the end.

In [ ]:
d = [3, 1, 0, 2]
q = [1, 0, 0, 1]

print("Document vector d =", d)
print("Query vector    q =", q)
print("\nGoal: Compute cosine similarity between d and q.")
print("First, we need to understand dot products, norms, and angles.")

---

## 3. Build — Core Concepts

We build up the fundamental inner-product-space operations one at a time,
starting with the dot product and working up to orthogonality and the
major inequalities.

### 3.1 The Dot Product

> **Definition (Larson, Sec. 5.1).** The **dot product** (or **inner product**)
> of two vectors $\mathbf{u}, \mathbf{v} \in \mathbb{R}^n$ is
>
> $$\mathbf{u} \cdot \mathbf{v} = \sum_{i=1}^{n} u_i v_i = u_1 v_1 + u_2 v_2 + \cdots + u_n v_n$$

The dot product takes two vectors of the same dimension and returns a **scalar**.
It is the most fundamental operation in inner product spaces.

In [ ]:
u = [2, -1, 3, 1]
v = [1, 4, -2, 5]

result = dot(u, v)

print(f"u = {u}")
print(f"v = {v}")
print()
print("Step-by-step:")
terms = [f"({u[i]})({v[i]})" for i in range(len(u))]
values = [u[i] * v[i] for i in range(len(u))]
print("  " + " + ".join(terms))
print("= " + " + ".join(f"{val}" for val in values))
print(f"= {result}")
print()
print(f"dot(u, v) = {result}")

### 3.2 Properties of the Dot Product

The dot product satisfies four key properties (Larson, Theorem 5.3):

| Property | Statement |
|---|---|
| Commutative | $\mathbf{u} \cdot \mathbf{v} = \mathbf{v} \cdot \mathbf{u}$ |
| Distributive | $\mathbf{u} \cdot (\mathbf{v} + \mathbf{w}) = \mathbf{u} \cdot \mathbf{v} + \mathbf{u} \cdot \mathbf{w}$ |
| Scalar factoring | $c(\mathbf{u} \cdot \mathbf{v}) = (c\mathbf{u}) \cdot \mathbf{v} = \mathbf{u} \cdot (c\mathbf{v})$ |
| Positive definite | $\mathbf{v} \cdot \mathbf{v} \geq 0$, with equality iff $\mathbf{v} = \mathbf{0}$ |

Let’s verify each computationally.

In [ ]:
u = [2, -1, 3]
v = [1, 4, -2]
w = [5, 0, 7]
c = 3.0

cu = [c * x for x in u]
cv = [c * x for x in v]
v_plus_w = [v[i] + w[i] for i in range(len(v))]
zero = [0, 0, 0]

print("1. Commutative: u . v == v . u")
print(f"   dot(u, v) = {dot(u, v)},  dot(v, u) = {dot(v, u)}")
print(f"   Equal? {abs(dot(u, v) - dot(v, u)) < EPSILON}")
print()

print("2. Distributive: u . (v + w) == u . v + u . w")
lhs = dot(u, v_plus_w)
rhs = dot(u, v) + dot(u, w)
print(f"   LHS = {lhs},  RHS = {rhs}")
print(f"   Equal? {abs(lhs - rhs) < EPSILON}")
print()

print("3. Scalar factoring: c(u . v) == (cu) . v == u . (cv)")
val1 = c * dot(u, v)
val2 = dot(cu, v)
val3 = dot(u, cv)
print(f"   c * dot(u,v) = {val1}")
print(f"   dot(cu, v)   = {val2}")
print(f"   dot(u, cv)   = {val3}")
print(f"   All equal? {abs(val1 - val2) < EPSILON and abs(val1 - val3) < EPSILON}")
print()

print("4. Positive definite: v . v >= 0, with equality iff v = 0")
print(f"   dot(v, v) = {dot(v, v)}  (>= 0? {dot(v, v) >= 0})")
print(f"   dot(0, 0) = {dot(zero, zero)}  (== 0? {dot(zero, zero) == 0})")

### 3.3 Norm (Length)

> **Definition (Larson, Sec. 5.1).** The **norm** (or **length**, or
> **Euclidean norm**) of a vector $\mathbf{v} \in \mathbb{R}^n$ is
>
> $$\|\mathbf{v}\| = \sqrt{\mathbf{v} \cdot \mathbf{v}} = \sqrt{v_1^2 + v_2^2 + \cdots + v_n^2}$$

The norm generalizes the Pythagorean theorem to $n$ dimensions.
In $\mathbb{R}^2$, $\|(a, b)\| = \sqrt{a^2 + b^2}$ is the familiar
distance from the origin.

In [ ]:
v = [3, 4]
print(f"v = {v}")
print(f"v . v = {v[0]}^2 + {v[1]}^2 = {dot(v, v)}")
print(f"||v|| = sqrt({dot(v, v)}) = {norm(v)}")
print()

w = [1, -2, 2, 4]
print(f"w = {w}")
print(f"w . w = " + " + ".join(f"{x}^2" for x in w) + f" = {dot(w, w)}")
print(f"||w|| = sqrt({dot(w, w)}) = {norm(w):.6f}")

### 3.4 Unit Vectors and Normalization

> **Definition (Larson, Sec. 5.1).** A vector $\mathbf{v}$ is a **unit vector** if
> $\|\mathbf{v}\| = 1$.
>
> **Normalization.** For any nonzero vector $\mathbf{v}$, the vector
>
> $$\hat{\mathbf{v}} = \frac{\mathbf{v}}{\|\mathbf{v}\|}$$
>
> is a unit vector in the same direction as $\mathbf{v}$.

Normalization preserves direction but rescales magnitude to exactly 1.
This is essential in applications like cosine similarity, where only
direction matters.

In [ ]:
v = [3, 4]
v_hat = normalize(v)

print(f"v     = {v}")
print(f"||v|| = {norm(v)}")
print(f"v_hat = v / ||v|| = [{v_hat[0]:.6f}, {v_hat[1]:.6f}]")
print(f"||v_hat|| = {norm(v_hat):.10f}")
print(f"Is unit vector? {abs(norm(v_hat) - 1.0) < EPSILON}")
print()

w = [1, -2, 2, 4]
w_hat = normalize(w)
print(f"w     = {w}")
print(f"||w|| = {norm(w):.6f}")
print(f"w_hat = [{', '.join(f'{x:.6f}' for x in w_hat)}]")
print(f"||w_hat|| = {norm(w_hat):.10f}")
print(f"Is unit vector? {abs(norm(w_hat) - 1.0) < EPSILON}")

### 3.5 Distance

> **Definition (Larson, Sec. 5.1).** The **distance** between two vectors
> $\mathbf{u}, \mathbf{v} \in \mathbb{R}^n$ is
>
> $$d(\mathbf{u}, \mathbf{v}) = \|\mathbf{u} - \mathbf{v}\| = \sqrt{\sum_{i=1}^{n} (u_i - v_i)^2}$$

This is the standard Euclidean distance. It equals zero if and only if
$\mathbf{u} = \mathbf{v}$.

In [ ]:
u = [1, 2, 3]
v = [4, 0, 1]

diff = [u[i] - v[i] for i in range(len(u))]
print(f"u = {u}")
print(f"v = {v}")
print(f"u - v = {diff}")
print(f"||u - v|| = sqrt(" + " + ".join(f"({d})^2" for d in diff) + ")")
print(f"         = sqrt({sum(d**2 for d in diff)})")
print(f"         = {distance(u, v):.6f}")
print()

print(f"Distance from v to itself: d(v, v) = {distance(v, v)}")
print(f"d(u, v) == d(v, u)? {abs(distance(u, v) - distance(v, u)) < EPSILON}")

### 3.6 Angle Between Vectors

> **Definition (Larson, Sec. 5.1).** The **angle** $\theta$ between two
> nonzero vectors $\mathbf{u}$ and $\mathbf{v}$ in $\mathbb{R}^n$ satisfies
>
> $$\cos\theta = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\|\,\|\mathbf{v}\|}, \qquad 0 \leq \theta \leq \pi$$

This formula works in **any dimension**, even though we can only visualize
angles in 2D or 3D. The `angle` function returns the result in radians.

In [ ]:
u = [1, 0]
v = [1, 1]

theta = angle(u, v)
cos_theta = dot(u, v) / (norm(u) * norm(v))

print(f"u = {u}")
print(f"v = {v}")
print(f"")
print(f"u . v     = {dot(u, v)}")
print(f"||u||     = {norm(u):.6f}")
print(f"||v||     = {norm(v):.6f}")
print(f"cos(theta) = {dot(u, v)} / ({norm(u):.4f} * {norm(v):.4f}) = {cos_theta:.6f}")
print(f"theta      = arccos({cos_theta:.6f}) = {theta:.6f} radians")
print(f"           = {math.degrees(theta):.2f} degrees")
print()
print(f"Expected: 45 degrees = pi/4 = {math.pi/4:.6f} radians")
print(f"Match? {abs(theta - math.pi/4) < EPSILON}")

### 3.7 Orthogonality

> **Definition (Larson, Sec. 5.1).** Two vectors $\mathbf{u}$ and $\mathbf{v}$
> are **orthogonal** if
>
> $$\mathbf{u} \cdot \mathbf{v} = 0$$

Geometrically, orthogonal vectors are perpendicular: the angle between them
is $\theta = \pi/2$ (90 degrees). When $\mathbf{u} \cdot \mathbf{v} = 0$,
the cosine formula gives $\cos\theta = 0$, hence $\theta = \pi/2$.

**Convention:** The zero vector is orthogonal to every vector.

In [ ]:
u = [1, 0, 0]
v = [0, 1, 0]
w = [1, 1, 0]

print(f"u = {u}")
print(f"v = {v}")
print(f"w = {w}")
print()

print(f"u . v = {dot(u, v)}")
print(f"are_orthogonal(u, v) = {are_orthogonal(u, v)}")
print()

print(f"u . w = {dot(u, w)}")
print(f"are_orthogonal(u, w) = {are_orthogonal(u, w)}")
print()

a = [2, -3, 1]
b = [1, 2, 4]
print(f"a = {a}")
print(f"b = {b}")
print(f"a . b = {dot(a, b)}")
print(f"are_orthogonal(a, b) = {are_orthogonal(a, b)}")
print("\nCheck: 2*1 + (-3)*2 + 1*4 = 2 - 6 + 4 = 0")

### 3.8 The Cauchy–Schwarz Inequality

> **Theorem (Larson, Theorem 5.5).** For any vectors
> $\mathbf{u}, \mathbf{v} \in \mathbb{R}^n$,
>
> $$|\mathbf{u} \cdot \mathbf{v}| \leq \|\mathbf{u}\|\,\|\mathbf{v}\|$$
>
> Equality holds if and only if $\mathbf{u}$ and $\mathbf{v}$ are
> **linearly dependent** (one is a scalar multiple of the other).

This inequality guarantees that the cosine formula always produces a value
in $[-1, 1]$, so the angle is always well-defined.

Let’s verify it empirically with 1000 random vector pairs.

In [ ]:
random.seed(42)
n_trials = 1000
dim = 5
violations = 0
closest_to_equality = float('inf')

for _ in range(n_trials):
    u = [random.uniform(-10, 10) for _ in range(dim)]
    v = [random.uniform(-10, 10) for _ in range(dim)]

    lhs = abs(dot(u, v))
    rhs = norm(u) * norm(v)
    gap = rhs - lhs

    if gap < -EPSILON:
        violations += 1
    if gap < closest_to_equality:
        closest_to_equality = gap

print("Cauchy-Schwarz Inequality: |u . v| <= ||u|| * ||v||")
print(f"Tested {n_trials} random pairs in R^{dim}")
print(f"Violations: {violations}")
print(f"Smallest gap (rhs - lhs): {closest_to_equality:.10f}")
print()

print("--- Equality case (linearly dependent vectors) ---")
u = [2, -1, 3]
v = [6, -3, 9]
lhs = abs(dot(u, v))
rhs = norm(u) * norm(v)
print(f"u = {u}")
print(f"v = 3u = {v}")
print(f"|u . v| = {lhs:.6f}")
print(f"||u|| * ||v|| = {rhs:.6f}")
print(f"Gap = {rhs - lhs:.2e}  (essentially zero — equality holds)")

### 3.9 The Triangle Inequality

> **Theorem (Larson, Theorem 5.6).** For any vectors
> $\mathbf{u}, \mathbf{v} \in \mathbb{R}^n$,
>
> $$\|\mathbf{u} + \mathbf{v}\| \leq \|\mathbf{u}\| + \|\mathbf{v}\|$$

In words: the length of the sum is at most the sum of the lengths.
Geometrically, the "direct path" from the origin to $\mathbf{u} + \mathbf{v}$
is never longer than going via $\mathbf{u}$ first and then adding $\mathbf{v}$.

The proof uses the Cauchy–Schwarz inequality:

$$\|\mathbf{u} + \mathbf{v}\|^2 = \|\mathbf{u}\|^2 + 2(\mathbf{u} \cdot \mathbf{v}) + \|\mathbf{v}\|^2 \leq \|\mathbf{u}\|^2 + 2\|\mathbf{u}\|\|\mathbf{v}\| + \|\mathbf{v}\|^2 = (\|\mathbf{u}\| + \|\mathbf{v}\|)^2$$

Let’s verify with 1000 random pairs.

In [ ]:
random.seed(99)
n_trials = 1000
dim = 5
violations = 0
closest_to_equality = float('inf')

for _ in range(n_trials):
    u = [random.uniform(-10, 10) for _ in range(dim)]
    v = [random.uniform(-10, 10) for _ in range(dim)]
    u_plus_v = [u[i] + v[i] for i in range(dim)]

    lhs = norm(u_plus_v)
    rhs = norm(u) + norm(v)
    gap = rhs - lhs

    if gap < -EPSILON:
        violations += 1
    if gap < closest_to_equality:
        closest_to_equality = gap

print("Triangle Inequality: ||u + v|| <= ||u|| + ||v||")
print(f"Tested {n_trials} random pairs in R^{dim}")
print(f"Violations: {violations}")
print(f"Smallest gap (rhs - lhs): {closest_to_equality:.10f}")
print()

print("--- Equality case (same direction) ---")
u = [2, -1, 3]
v = [4, -2, 6]
u_plus_v = [u[i] + v[i] for i in range(len(u))]
lhs = norm(u_plus_v)
rhs = norm(u) + norm(v)
print(f"u = {u}")
print(f"v = 2u = {v}")
print(f"||u + v|| = {lhs:.6f}")
print(f"||u|| + ||v|| = {rhs:.6f}")
print(f"Gap = {rhs - lhs:.2e}  (essentially zero — equality holds)")

### 3.10 Solving the Search Problem — Cosine Similarity

Now we have all the tools to answer the motivating question.
The **cosine similarity** between the document vector $\mathbf{d}$ and
query vector $\mathbf{q}$ is:

$$\cos\theta = \frac{\mathbf{d} \cdot \mathbf{q}}{\|\mathbf{d}\|\,\|\mathbf{q}\|}$$

Values close to 1 indicate high similarity; values close to 0 indicate
the document has little relevance to the query.

In [ ]:
d = [3, 1, 0, 2]
q = [1, 0, 0, 1]

d_dot_q = dot(d, q)
norm_d = norm(d)
norm_q = norm(q)
cos_sim = d_dot_q / (norm_d * norm_q)
theta = angle(d, q)

print("Search Engine Similarity")
print("=" * 40)
print(f"Document  d = {d}")
print(f"Query     q = {q}")
print()
print(f"d . q     = 3*1 + 1*0 + 0*0 + 2*1 = {d_dot_q}")
print(f"||d||     = sqrt(9 + 1 + 0 + 4) = sqrt(14) = {norm_d:.6f}")
print(f"||q||     = sqrt(1 + 0 + 0 + 1) = sqrt(2)  = {norm_q:.6f}")
print()
print(f"cos(theta) = {d_dot_q} / ({norm_d:.4f} * {norm_q:.4f})")
print(f"           = {d_dot_q} / {norm_d * norm_q:.6f}")
print(f"           = {cos_sim:.6f}")
print()
print(f"theta      = {theta:.6f} radians = {math.degrees(theta):.2f} degrees")
print()
print(f"Similarity score: {cos_sim:.4f}")
print(f"Interpretation: The document is reasonably similar to the query.")
print(f"(A score of 1.0 = perfect match, 0.0 = no overlap)")

---

## 4. Verify — Cross-Check with NumPy

Our from-scratch implementations should match NumPy exactly.
We verify each function by comparing against NumPy equivalents.

In [ ]:
def check(ours, theirs, label):
    """Compare a scalar result against NumPy."""
    match = abs(ours - theirs) < 1e-9
    status = "PASS" if match else "FAIL"
    print(f"[{status}] {label}")
    if not match:
        print(f"  Ours:  {ours}")
        print(f"  NumPy: {theirs}")
    return match


def check_vec(ours, theirs, label):
    """Compare a vector result against NumPy."""
    match = all(abs(a - b) < 1e-9 for a, b in zip(ours, theirs))
    status = "PASS" if match else "FAIL"
    print(f"[{status}] {label}")
    if not match:
        print(f"  Ours:  {ours}")
        print(f"  NumPy: {list(theirs)}")
    return match

In [ ]:
u = [2, -1, 3, 1, 7]
v = [1, 4, -2, 5, 0]
u_np = np.array(u, dtype=float)
v_np = np.array(v, dtype=float)

print("=" * 50)
print("VERIFICATION: Dot product")
print("=" * 50)
check(dot(u, v), np.dot(u_np, v_np), "dot(u, v) matches np.dot")
print()

print("=" * 50)
print("VERIFICATION: Norm")
print("=" * 50)
check(norm(u), np.linalg.norm(u_np), "norm(u) matches np.linalg.norm")
check(norm(v), np.linalg.norm(v_np), "norm(v) matches np.linalg.norm")
print()

print("=" * 50)
print("VERIFICATION: Normalize")
print("=" * 50)
check_vec(normalize(u), u_np / np.linalg.norm(u_np), "normalize(u) matches np unit vector")
check_vec(normalize(v), v_np / np.linalg.norm(v_np), "normalize(v) matches np unit vector")
print()

print("=" * 50)
print("VERIFICATION: Distance")
print("=" * 50)
check(distance(u, v), np.linalg.norm(u_np - v_np), "distance(u, v) matches np.linalg.norm(u-v)")
print()

print("=" * 50)
print("VERIFICATION: Angle")
print("=" * 50)
np_cos = np.dot(u_np, v_np) / (np.linalg.norm(u_np) * np.linalg.norm(v_np))
np_angle = np.arccos(np.clip(np_cos, -1.0, 1.0))
check(angle(u, v), np_angle, "angle(u, v) matches np.arccos formula")

In [ ]:
print("=" * 50)
print("VERIFICATION: Cauchy-Schwarz for 1000 random pairs")
print("=" * 50)

random.seed(77)
cs_violations = 0

for _ in range(1000):
    u = [random.gauss(0, 5) for _ in range(8)]
    v = [random.gauss(0, 5) for _ in range(8)]
    u_np = np.array(u)
    v_np = np.array(v)

    our_lhs = abs(dot(u, v))
    our_rhs = norm(u) * norm(v)

    np_lhs = abs(np.dot(u_np, v_np))
    np_rhs = np.linalg.norm(u_np) * np.linalg.norm(v_np)

    if our_rhs - our_lhs < -EPSILON:
        cs_violations += 1

    if abs(our_lhs - np_lhs) > 1e-9 or abs(our_rhs - np_rhs) > 1e-9:
        cs_violations += 1

status = "PASS" if cs_violations == 0 else "FAIL"
print(f"[{status}] Cauchy-Schwarz holds for all 1000 pairs, values match NumPy")
print(f"  Violations: {cs_violations}")

---

## 5. Visualize

### 5.1 Two Vectors with Angle Labeled

We plot two 2D vectors from the origin and draw an arc showing the
angle between them, labeled with the computed value.

In [ ]:
u = [3, 1]
v = [1, 2]

theta = angle(u, v)
angle_u = math.atan2(u[1], u[0])
angle_v = math.atan2(v[1], v[0])

fig, ax = plt.subplots(1, 1, figsize=(7, 6))

ax.annotate('', xy=u, xytext=(0, 0),
            arrowprops=dict(arrowstyle='->', color='steelblue', lw=2.5))
ax.annotate('', xy=v, xytext=(0, 0),
            arrowprops=dict(arrowstyle='->', color='tomato', lw=2.5))

ax.text(u[0] + 0.1, u[1] + 0.1, f'u = ({u[0]}, {u[1]})',
        fontsize=12, color='steelblue', fontweight='bold')
ax.text(v[0] + 0.1, v[1] + 0.1, f'v = ({v[0]}, {v[1]})',
        fontsize=12, color='tomato', fontweight='bold')

start_deg = math.degrees(min(angle_u, angle_v))
end_deg = math.degrees(max(angle_u, angle_v))
arc = patches.Arc((0, 0), 1.2, 1.2, angle=0,
                   theta1=start_deg, theta2=end_deg,
                   color='green', lw=2, linestyle='--')
ax.add_patch(arc)

mid_angle = (angle_u + angle_v) / 2
label_r = 0.9
ax.text(label_r * math.cos(mid_angle), label_r * math.sin(mid_angle),
        f'$\\theta$ = {math.degrees(theta):.1f}\u00b0',
        fontsize=12, color='green', fontweight='bold',
        ha='center', va='center')

ax.set_xlim(-0.5, 4)
ax.set_ylim(-0.5, 3)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.axhline(0, color='gray', linewidth=0.5)
ax.axvline(0, color='gray', linewidth=0.5)
ax.set_title(f'Angle Between u and v: {math.degrees(theta):.1f}\u00b0 ({theta:.4f} rad)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')

plt.tight_layout()
plt.show()

### 5.2 Orthogonal Projection Preview

Although orthogonal projection is formally covered in Section 5.3,
we can preview the geometry here. The **projection** of $\mathbf{v}$
onto $\mathbf{u}$ is:

$$\text{proj}_{\mathbf{u}}\,\mathbf{v} = \frac{\mathbf{v} \cdot \mathbf{u}}{\mathbf{u} \cdot \mathbf{u}}\,\mathbf{u}$$

The projection drops a perpendicular from $\mathbf{v}$ onto the line
spanned by $\mathbf{u}$. The vector from the projection to $\mathbf{v}$
is orthogonal to $\mathbf{u}$.

In [ ]:
u = [4, 1]
v = [2, 3]

scale = dot(v, u) / dot(u, u)
proj = [scale * x for x in u]
perp = [v[i] - proj[i] for i in range(len(v))]

fig, ax = plt.subplots(1, 1, figsize=(8, 6))

t_vals = [-0.3, 1.5]
line_x = [t * u[0] for t in t_vals]
line_y = [t * u[1] for t in t_vals]
ax.plot(line_x, line_y, '--', color='gray', alpha=0.5, linewidth=1,
        label='span(u)')

ax.annotate('', xy=u, xytext=(0, 0),
            arrowprops=dict(arrowstyle='->', color='steelblue', lw=2.5))
ax.text(u[0] + 0.1, u[1] + 0.15, f'u = ({u[0]}, {u[1]})',
        fontsize=11, color='steelblue', fontweight='bold')

ax.annotate('', xy=v, xytext=(0, 0),
            arrowprops=dict(arrowstyle='->', color='tomato', lw=2.5))
ax.text(v[0] + 0.1, v[1] + 0.15, f'v = ({v[0]}, {v[1]})',
        fontsize=11, color='tomato', fontweight='bold')

ax.annotate('', xy=proj, xytext=(0, 0),
            arrowprops=dict(arrowstyle='->', color='green', lw=2))
ax.text(proj[0] + 0.1, proj[1] - 0.4,
        f'proj = ({proj[0]:.2f}, {proj[1]:.2f})',
        fontsize=10, color='green', fontweight='bold')

ax.annotate('', xy=v, xytext=proj,
            arrowprops=dict(arrowstyle='->', color='purple', lw=1.5, linestyle='dashed'))
mid = [(proj[i] + v[i]) / 2 for i in range(2)]
ax.text(mid[0] - 0.7, mid[1] + 0.1, 'v - proj\n(perp to u)',
        fontsize=9, color='purple', style='italic')

sq_size = 0.2
perp_unit = normalize(perp)
u_unit = normalize(u)
sq_corner = proj
sq = plt.Polygon([
    sq_corner,
    [sq_corner[0] + sq_size * perp_unit[0], sq_corner[1] + sq_size * perp_unit[1]],
    [sq_corner[0] + sq_size * perp_unit[0] - sq_size * u_unit[0],
     sq_corner[1] + sq_size * perp_unit[1] - sq_size * u_unit[1]],
    [sq_corner[0] - sq_size * u_unit[0], sq_corner[1] - sq_size * u_unit[1]],
], closed=True, fill=False, edgecolor='purple', linewidth=1.5)
ax.add_patch(sq)

ax.set_xlim(-1, 5.5)
ax.set_ylim(-0.5, 4)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.axhline(0, color='gray', linewidth=0.5)
ax.axvline(0, color='gray', linewidth=0.5)
ax.set_title('Orthogonal Projection of v onto u (Preview)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')

orth_check = dot(perp, u)
ax.text(0.02, 0.02,
        f'Verification: (v - proj) . u = {orth_check:.2e} (orthogonal)',
        transform=ax.transAxes, fontsize=9, style='italic',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.tight_layout()
plt.show()

---

## 6. Exercises

Test your understanding with the following problems. Write your solutions
in the code cells provided.

### Exercise 1 (Easy)

Given $\mathbf{u} = (1, 3, -2, 4)$ and $\mathbf{v} = (2, -1, 0, 3)$:

1. Compute the angle between $\mathbf{u}$ and $\mathbf{v}$ using the `angle` function.
2. Convert the result from **radians to degrees** using `math.degrees`.
3. Print both the radian and degree values.

In [ ]:
# YOUR CODE HERE


### Exercise 2 (Medium)

Given $\mathbf{u} = (1, 2, 3)$ and $\mathbf{v} = (4, 5, 6)$, find a vector
$\mathbf{w} \in \mathbb{R}^3$ that is **orthogonal to both** $\mathbf{u}$ and $\mathbf{v}$.

*Strategy:* A vector $\mathbf{w} = (w_1, w_2, w_3)$ is orthogonal to both $\mathbf{u}$ and $\mathbf{v}$
if and only if:

$$\mathbf{u} \cdot \mathbf{w} = 0 \quad \text{and} \quad \mathbf{v} \cdot \mathbf{w} = 0$$

This gives two equations in three unknowns:

$$\begin{cases} w_1 + 2w_2 + 3w_3 = 0 \\ 4w_1 + 5w_2 + 6w_3 = 0 \end{cases}$$

Solve this system using `solve` from `linalg_core.elimination` (you already imported it).
Verify that the solution is orthogonal to both $\mathbf{u}$ and $\mathbf{v}$ using `are_orthogonal`.

In [ ]:
# YOUR CODE HERE


### Exercise 3 (Challenge)

Write a function `cosine_similarity_matrix(vectors)` that takes a list of $k$ vectors
(each a list of floats of the same length) and returns a $k \times k$ **similarity matrix**
$S$ where:

$$S_{ij} = \frac{\mathbf{v}_i \cdot \mathbf{v}_j}{\|\mathbf{v}_i\|\,\|\mathbf{v}_j\|}$$

Test it with the following four "document" vectors and print the resulting matrix.
Verify that $S_{ii} = 1$ for all $i$ and $S_{ij} = S_{ji}$ for all $i, j$.

```python
docs = [
    [3, 1, 0, 2],   # doc 0
    [1, 0, 0, 1],   # doc 1 (the query from the motivation)
    [0, 0, 4, 1],   # doc 2
    [3, 1, 0, 2],   # doc 3 (identical to doc 0)
]
```

*Bonus:* Which pair of distinct documents is most similar? Which is least?

In [ ]:
# YOUR CODE HERE


---

## Summary

| Concept | Formula | Key Takeaway |
|---|---|---|
| **Dot product** | $\mathbf{u} \cdot \mathbf{v} = \sum u_i v_i$ | Measures alignment; returns a scalar |
| **Norm** | $\|\mathbf{v}\| = \sqrt{\mathbf{v} \cdot \mathbf{v}}$ | Generalizes Pythagorean length to $\mathbb{R}^n$ |
| **Unit vector** | $\hat{\mathbf{v}} = \mathbf{v} / \|\mathbf{v}\|$ | Direction without magnitude |
| **Distance** | $d(\mathbf{u}, \mathbf{v}) = \|\mathbf{u} - \mathbf{v}\|$ | Standard Euclidean distance |
| **Angle** | $\cos\theta = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\|\,\|\mathbf{v}\|}$ | Works in any dimension |
| **Orthogonality** | $\mathbf{u} \cdot \mathbf{v} = 0$ | Perpendicular; angle is $\pi/2$ |
| **Cauchy–Schwarz** | $|\mathbf{u} \cdot \mathbf{v}| \leq \|\mathbf{u}\|\,\|\mathbf{v}\|$ | Guarantees $\cos\theta \in [-1, 1]$ |
| **Triangle inequality** | $\|\mathbf{u} + \mathbf{v}\| \leq \|\mathbf{u}\| + \|\mathbf{v}\|$ | Direct path is never longer |
| **Cosine similarity** | $\frac{\mathbf{d} \cdot \mathbf{q}}{\|\mathbf{d}\|\,\|\mathbf{q}\|}$ | Foundation of document search |

**Next:** In Section 5.2, we extend these ideas to general **inner product spaces** where
the inner product can be defined by a symmetric positive-definite matrix.